In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import types as T
from pyspark.sql import functions as F

import os

import sys
import glob

In [2]:
os.environ.pop("SPARK_HOME", None)
os.environ.pop("CLASSPATH", None)

In [3]:
import sys
import glob

PROJECT_ROOT = "/home/yogavarman/Projects/FoodChain"
FUNCTIONS_ZIP = f"{PROJECT_ROOT}/foodchain_functions.zip"

sys.path.insert(0, PROJECT_ROOT)

jars = glob.glob("/opt/spark/jars/*.jar")

spark = (
    SparkSession.builder
    .appName("Clean_users")
    .config("spark.jars", ",".join(jars))
    .config("spark.submit.pyFiles", FUNCTIONS_ZIP)
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/26 14:41:43 WARN Utils: Your hostname, 2640L, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/26 14:41:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/08/26 14:41:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/yogavarman/venvs/global_env/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [ ]:
# import sys
# sys.path.insert(0, "/home/yogavarman/Projects/FoodChain")

# from Config.db import JDBC_URL, DB_PROPERTIES, DATABASE_URL, get_conn

# from Functions.LogInFun import generate_password, hash_password

# from pyspark.sql import functions as F
# from pyspark.sql import types as T

In [4]:
import pandas as pd
from pathlib import Path

folderpath = Path("/home/yogavarman/Projects/FoodChain/DataSet/RawData")
filename = "users"

xlsx_path = folderpath / f"{filename}.xlsx"
csv_path = folderpath / f"{filename}.csv"

# Read the specific sheet
df_excel = pd.read_excel(xlsx_path, sheet_name="users")  # or sheet_name=0


# Write to CSV
df_excel.to_csv(csv_path, index=False)

print("CSV created at:", csv_path)

CSV created at: /home/yogavarman/Projects/FoodChain/DataSet/RawData/users.csv


In [60]:
df =  (
    spark.read
    .format("csv")
    .option("header",True)
    .option("inferSchema",False)
    .load(str(csv_path))
)


In [61]:
df.count()

100000

In [62]:

df = df.dropDuplicates(["name"])

In [63]:
df.count()

71251

In [64]:
from pyspark.sql.window import Window

window = Window.orderBy(F.monotonically_increasing_id())

df = df.withColumn(
    "user_id",
    (F.row_number().over(window) + 202608260).cast("long")
)

In [65]:
providers = ["gmail.com", "outlook.com", "zoho.com", "hotmail.com", "yahoo.com"]
df = (df
    .withColumn(
        "email",
        F.concat(
            F.lower(
                F.regexp_replace(
                    F.trim(F.col("name")),
                    r"\s+",
                    "."
                )
            ),
            F.lit("@"),
            F.element_at(
                F.array(*[F.lit(x) for x in providers]),
                (F.rand() * len(providers)).cast("int") + 1
            )
        )
    )
)

In [66]:
df = df.withColumn(
    "dob",
    F.add_months(F.current_date(), -F.col("age") * 12)
)

In [67]:
from Config.db import (
    JDBC_URL,
    DB_PROPERTIES,
    DATABASE_URL,
    get_conn
)

from Functions.LogInFun import (
    generate_password,
    hash_password
)

In [68]:



# Create UDF
generate_password_udf = F.udf(
    generate_password,
    F.StringType()
)


# Generate random password
df = df.withColumn(
    "password",
    generate_password_udf()
)


# Generate SHA-256 password hash
df = df.withColumn(
    "password_hash",
    F.sha2(F.col("password"), 256)
)

/home/yogavarman/venvs/global_env/lib/python3.14/site-packages/pyspark/sql/udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [69]:
df = (
    df
    .withColumn("first_name", F.split(F.trim(F.col("name")), r"\s+").getItem(0))
    .withColumn("last_name", F.split(F.trim(F.col("name")), r"\s+").getItem(1))
)

In [70]:
df=df.withColumn("username",F.col("user_id"))


In [71]:
df.printSchema()

root
 |-- user_id: long (nullable = false)
 |-- name: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Marital Status: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- email: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- password: string (nullable = true)
 |-- password_hash: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- username: long (nullable = false)



In [72]:
df = df.select("user_id", "username", "password_hash", "first_name", "last_name", 
               "gender", "email", "dob", "password")
df = df.withColumnRenamed("dob", "date_of_birth")
df=df.withColumn("date_of_birth", F.date_format(F.col("date_of_birth"), "yyyy-MM-dd"))

In [73]:
df = df.withColumn(
    "date_of_birth",
    F.to_date(F.col("date_of_birth"), "yyyy-MM-dd")
)
df.printSchema()

root
 |-- user_id: long (nullable = false)
 |-- username: long (nullable = false)
 |-- password_hash: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- email: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- password: string (nullable = true)



In [74]:

df.write.jdbc(
    url=JDBC_URL,
    table="foodchain.users",
    mode="append",
    properties=DB_PROPERTIES
)

26/08/26 14:52:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/26 14:52:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/26 14:52:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/26 14:52:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/26 14:52:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/26 14:52:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/26 1

In [75]:
spark.stop()